# Iris Decision Lab

This notebook adapts the Iris decision-tree example from chapter 6 of
Aurélien Géron's *Hands-On Machine Learning with Scikit-Learn, Keras &
TensorFlow* (3rd edition, Apache-2.0, source notebook
[`06_decision_trees.ipynb`](https://github.com/ageron/handson-ml3/blob/e707c2d659abafb9b1f9fd927907619a128db8d7/06_decision_trees.ipynb)).

Instead of a single `DecisionTreeClassifier`, the lab compares three models —
a decision tree, a random forest, and logistic regression — on a stratified
70/30 held-out split of the 150-sample Iris dataset. The workflow is split
into reusable stages (import, split, train, evaluate, persist) that the
Streamlit app (`app.py`) and the AGILAB workflow reuse.

The notebook is self-contained: it resolves the project directory at runtime
(via the `PROJECT_ROOT` variable when provided, otherwise the current
working directory), imports `models.py` from there, and writes
`metrics.json` to the **current working directory** so it can run from any
location without downloads, magics, or pip installs.

Because the dataset is tiny (150 rows, 4 features), accuracy differences
between models are small and split-dependent; the results below describe
this one split, not general model quality.

In [ ]:
"""Stage 1 — import: model factory, data, and reusable workflow stages."""
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

_project_root = Path(globals().get("PROJECT_ROOT", Path.cwd())).resolve()
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from models import build_models, load_data  # noqa: E402

SPLIT_SEED = 42
TEST_SIZE = 0.3


def split_data(seed: int = SPLIT_SEED):
    """Split the full Iris frame into a stratified train/test pair."""
    X, y, names = load_data()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=seed
    )
    return X_train, X_test, y_train, y_test, names


def train_models(max_depth: int = 3, seed: int = SPLIT_SEED):
    """Fit every candidate model from `models.build_models` on the train split."""
    X_train, X_test, y_train, y_test, names = split_data(seed)
    fitted = {}
    for name, estimator in build_models(max_depth=max_depth, seed=seed).items():
        fitted[name] = estimator.fit(X_train, y_train)
    return fitted, X_test, y_test, names


def evaluate_models(fitted: dict, X_test, y_test) -> list:
    """Score every model on the held-out split as JSON-ready rows."""
    rows = []
    for name, model in fitted.items():
        score = float(accuracy_score(y_test, model.predict(X_test)))
        rows.append({"model": name, "accuracy": round(score, 4)})
    return rows


def write_metrics(rows: list, path: Path = Path("metrics.json")) -> Path:
    """Persist the comparison to metrics.json in the current directory."""
    path.write_text(json.dumps(rows, indent=2, sort_keys=True))
    return path

## Run the workflow

The cell below chains the stages — `train_models` (split + fit) and
`evaluate_models` (held-out scoring) — and persists the result with
`write_metrics`. Each row of `metrics.json` records one model's held-out
accuracy. Re-run with different `max_depth`/`seed` values to see how the
ranking shifts; with 150 samples the differences stay small.

In [ ]:
"""Stage 2 — train + evaluate: run the pipeline and write metrics.json."""
fitted, X_test, y_test, names = train_models(max_depth=3, seed=SPLIT_SEED)
rows = evaluate_models(fitted, X_test, y_test)
out = write_metrics(rows)

comparison = pd.DataFrame(rows).sort_values("accuracy", ascending=False).reset_index(drop=True)
best = comparison.loc[0, "model"]
print(f"metrics written to {out}")
print(comparison.to_string(index=False))
print(f"Best on this held-out split: {best}")
print("Note: 150-sample dataset, single split — treat rankings as indicative only.")